# Rung 1/4/5 — standard-scale induction run and the head-validation cascade (GPU)

Issue #169. One Kaggle/Colab session, decided **before** seeing results:

| Step | What runs | Why |
|------|-----------|-----|
| 0 | **Pilot**: 20 epochs of the standard config, timed | Measures this GPU's real throughput (and the fresh-batch data generation, which is pure Python) so the session budget is planned from a measurement instead of an estimate |
| 1 | `exp1 --standard --seeds 0,1,2` (stateful: checkpoint every 250 epochs, resume, detect every 50, stop on confirmed) | The canonical MP-10 config the issue names — vocab 2048, seq 64, d_model 64, 2L/4H, 3000 epochs, fresh batches. 384k optimizer steps per seed |
| 2 | **Only if** step 1 confirmed no head: `exp1 --scale large --seeds 0,1,2` | The pre-registered escalation (vocab 2048, seq 128, d_model 256, 4L/8H, 1500 epochs). Decided in advance, not after seeing the numbers |
| 3 | `exp4 --from-checkpoint <exp1's checkpoint dir>` | Activation patching + head ablation + path patching **on the very model Rung 1 trained** — one model, not two. The unit-tested-only path patching gets its first end-to-end run |
| 4 | `exp5 --activations-from <rolling checkpoint>` | The first SAE on real activations from a model with a *confirmed* head (the harvest fix landed 2026-10-12; rolling checkpoints now load) |

**Falsifiers, fixed in advance:**
- The detector is planted-tested (`tests/test_planted_induction_detector.py`) *before* this run: the copy offset `q-(L-1)` must fire, the previous-token offset `q-1` (what the pre-2026-10-19 detector measured and called "diag+1") must not. A "0 heads" here is a property of the model, not of untested detection code.
- If step 1 AND step 2 both find no head, that is the result: report it with manifests, do not tune until one appears.
- Every step writes `--manifest-path` under `results/`; nothing is edited by hand afterwards.
- The **content-vs-position probe** runs on every model: on this periodic task a copy-by-content circuit and a fixed-offset circuit are the same function, so only the permuted-batch probe can say which one was learned.

**Before running:** enable a GPU (Kaggle: Settings -> Accelerator -> GPU T4/P100, Internet On; Colab: Runtime -> Change runtime type -> GPU). Sessions are ~9h; every long step is resumable — re-run its cell after a disconnect.

In [ ]:
BRANCH = "main"  # a pushed branch or tag — pin it so the manifest's git_sha is a real commit
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
# The GPU images already ship the CUDA torch stack plus numpy, matplotlib, tqdm and
# pyyaml. `uv sync --frozen` would re-resolve the ~5GB CUDA wheel set into the
# notebook's limited working disk and die mid-download — exactly how the first
# grokking kernel errored (2026-10-14: a 64MB partial venv, torch never landed).
# Use the preinstalled stack and fill only a real gap; the manifests record the
# torch/python versions actually used.
import importlib.util
import sys

missing = [
    m
    for m in ("torch", "numpy", "matplotlib", "tqdm", "yaml")
    if importlib.util.find_spec(m) is None
]
if missing:
    !pip install -q {' '.join(missing)}

import torch

assert torch.cuda.is_available(), "no GPU — enable one before running"
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| py", sys.version.split()[0])

## Step 0 — Pilot: measure this GPU before spending the session

Twenty epochs of the **standard config** (same shape, same fresh-batch regime, same detection cadence), timed. The output is the number the budget has to be planned from: the old notebook estimated "~2h" per run, which was never measured.

If the projection does not fit the session, the fix is **one seed per session** (set `SEEDS` below to a single seed), never a shorter budget: a truncated run is exactly how this repository produced under-trained false negatives.

In [ ]:
# 20 epochs of the standard config: 20 * 128 = 2560 optimizer steps.
import os, time
t0 = time.time()
!python -m src.experiments.exp1_induction_heads --vocab-size 2048 --seq-len 64 --d-model 64 --n-layers 2 --n-heads 4 --epochs 20 --batch-size 64 --num-train 8192 --batches fresh --seeds 0 --detect-every 10 --manifest-path results/probe_pilot_standard.json
pilot_seconds = time.time() - t0
steps = 20 * 128
rate = steps / max(pilot_seconds, 1e-6)
full = 3000 * 128
print(f"pilot: {pilot_seconds:.1f}s for {steps} steps -> {rate:.1f} steps/s")
print(f"projected standard run: {full / max(rate, 1e-6) / 60:.1f} min per seed")
print(f"projected 3 seeds:       {3 * full / max(rate, 1e-6) / 3600:.2f} h")

## Step 1 — The standard-scale run

The canonical config the issue names. `SEEDS` is an environment variable so a short session can run one seed at a time; the checkpoints and the per-seed manifest are the same either way.

`--stop-on-confirmed` ends a seed as soon as a head is confirmed at two consecutive checks: the positive case is bounded by the mechanism, while a negative seed still runs the **full** 384k-step budget.

In [ ]:
import os
os.environ["SEEDS"] = "0,1,2"  # set to a single seed (e.g. "0") if the session is short
!python -m src.experiments.exp1_induction_heads --standard --batches fresh --seeds $SEEDS --checkpoint-dir checkpoints/induction_standard --checkpoint-every 250 --resume --detect-every 50 --stop-on-confirmed --manifest-path results/induction_standard.json

In [ ]:
import json

m = json.load(open("results/induction_standard.json"))
for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
    print(
        f"seed {seed}: heads={int(metrics['total_induction_heads'])} "
        f"(screen {int(metrics['total_mass_heads'])}) | "
        f"val acc {metrics['final_val_acc']:.4f} | "
        f"induction mass {metrics['peak_induction_mass']:.3f} | "
        f"prev-token mass {metrics['peak_prev_token_mass']:.3f} | "
        f"composition {metrics['k_composition_score']:.3f} | "
        f"epochs {int(metrics['epochs_run'])} | "
        f"probe content {metrics['probe_content_mass']:.3f} vs period {metrics['probe_period_mass']:.3f}"
    )
FOUND_HEAD = any(int(x["total_induction_heads"]) > 0 for x in m["per_seed_metrics"])
CONTENT_BASED = any(float(x["probe_content_mass"]) > float(x["probe_period_mass"]) for x in m["per_seed_metrics"])
print(f"\nstandard scale found a head: {FOUND_HEAD}")
print(f"copy mechanism is content-based (not a fixed offset): {CONTENT_BASED}")

## Step 2 — The pre-registered escalation (only if step 1 confirmed nothing)

Larger model, longer sequences, fewer epochs: `--scale large` (vocab 2048, seq 128, d_model 256, 4L/8H, 1500 epochs = 192k steps per seed). The rule was fixed before step 1 ran: **escalate only if the standard config confirmed no head at its full budget.**

In [ ]:
if not FOUND_HEAD:
    !python -m src.experiments.exp1_induction_heads --scale large --batches fresh --seeds $SEEDS --checkpoint-dir checkpoints/induction_large --checkpoint-every 500 --resume --detect-every 50 --stop-on-confirmed --manifest-path results/induction_scale_large.json
else:
    print("A head was confirmed at standard scale — the escalation is skipped by the pre-registered rule.")

In [ ]:
import json, os

if FOUND_HEAD:
    m = json.load(open("results/induction_standard.json"))
else:
    m = json.load(open("results/induction_scale_large.json"))
for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
    print(f"exp4 seed {seed}: heads={int(metrics['total_induction_heads'])} | "
          f"ablation {metrics['mean_ablation_effect']:.3f} | "
          f"path patch {metrics['mean_path_patch_effect']:.3f} | "
          f"activation recovery {metrics['mean_activation_recovery']:.3f} | "
          f"checkpoint epoch {int(metrics['checkpoint_epoch'])}")

# Rung 4 analyses the SAME model Rung 1 trained: the checkpoint is loaded,
# not retrained. That is what makes the cascade one model instead of two.
if FOUND_HEAD:
    !python -m src.experiments.exp4_circuit_patching --from-checkpoint checkpoints/induction_standard --source-vocab-size 2048 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --seeds $SEEDS --manifest-path results/exp4_standard.json
else:
    !python -m src.experiments.exp4_circuit_patching --from-checkpoint checkpoints/induction_large --source-vocab-size 2048 --source-seq-len 128 --source-d-model 256 --source-n-layers 4 --source-n-heads 8 --seeds $SEEDS --manifest-path results/exp4_scale_large.json

In [ ]:
# Step 4 — the SAE on real activations from a checkpoint with a confirmed head.
import glob
import os

def first_head_seed():
    path = "results/induction_standard.json" if FOUND_HEAD else "results/induction_scale_large.json"
    if not os.path.exists(path):
        return None
    m = json.load(open(path))
    for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
        if int(metrics["total_induction_heads"]) > 0:
            return seed
    return None

seed = first_head_seed()
if seed is None:
    print("No seed confirmed an induction head — the exp5 real-activation run has no")
    print("confirmed-head checkpoint to harvest from. This is the honest outcome;")
    print("the synthetic SAE baseline already reproduces.")
else:
    ckpt_dir = "checkpoints/induction_standard" if FOUND_HEAD else "checkpoints/induction_large"
    ckpt = os.path.join(ckpt_dir, f"exp1_checkpoint_seed{seed}.pt")
    if FOUND_HEAD:
        src = "--source-vocab-size 2048 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4"
        d_model = 64
    else:
        src = "--source-vocab-size 2048 --source-seq-len 128 --source-d-model 256 --source-n-layers 4 --source-n-heads 8"
        d_model = 256
    !python -m src.experiments.exp5_sae_dashboard --activations-from $ckpt $src --d-model $d_model --seeds 42 --manifest-path results/exp5_real_activations.json

In [ ]:
# Collect every manifest this session produced, then download.
import glob
import os

manifests = sorted(glob.glob("results/induction_*.json") + glob.glob("results/exp4_*.json") + glob.glob("results/exp5_real_activations.json"))
manifests = [p for p in manifests if "probe_" not in p]
print("manifests:", manifests)
!zip -j induction_run_results.zip $manifests
try:
    from google.colab import files
    files.download("induction_run_results.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")

## After downloading

1. Unzip into a scratch dir, then pin: `python scripts/pin_remote_run.py --from <dir>` — it validates provenance (git sha must exist locally, clean tree, seeds/aggregate contract) and installs only what passes.
2. `make verify-claims` must stay green — new manifests need their `<!-- manifest: results/... -->` tags next to the numbers they back.
3. Report exactly what happened, including "no head at either scale": the answer to issue #169 is data, either way.
4. If a head WAS confirmed: update the Rung 1 section of `portfolio/RESULTS.md` with the new numbers — the "0 / 8 heads" claim it carries was produced by a detector that measured the previous-token offset, and it is retracted by this run.
5. If the probe says the copy is **positional** rather than content-based, say so: on this task the two are indistinguishable in-distribution, and the wording in RESULTS.md and the paper has to follow the measurement.